# Preprocessing

## Imports

In [49]:
import pandas as pd
import random
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

from utils.data.queries import execute_query

## Random Seed

In [46]:
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## Load Data

In [37]:
df = execute_query(
    "SELECT name, group_id, group_name FROM ml_dataset LIMIT :limit",
    limit=50000
)

## Separator Normalization

In [38]:
df["name_processed"] = df["name"].str.replace(r"[-_/|)(]+", " ", regex=True)

## Whitespace Normalization

In [39]:
df["name_processed"] = df["name_processed"].str.strip()
df["name_processed"] = df["name_processed"].str.replace(r"\s+", " ", regex=True)

## Train-Validation-Test Split

In [63]:
test_size = 0.1
val_size = 0.2

In [64]:
target_col = "group_id"

train_df, temp_df = train_test_split(
    df,
    test_size=test_size + val_size,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=df[target_col]
)

# 3. second split: val + test
val_ratio = val_size / (test_size + val_size)

val_df, test_df = train_test_split(
    temp_df,
    test_size=1 - val_ratio,
    random_state=RANDOM_STATE,
    stratify=temp_df[target_col]
)

In [68]:
splits = {
    "train": train_df,
    "val": val_df,
    "test": test_df
}


print("Dataset sizes:")
for name, df_ in splits.items():
    print(f"{name:5s}: {len(df_):,}")

total = len(train_df) + len(val_df) + len(test_df)
print(f"\nTotal: {total:,}\n")

# 2. Class coverage check
print("Class coverage (number of classes per split):")
all_classes = set(train_df[target_col].unique())

for name, df_ in splits.items():
    classes = set(df_[target_col].unique())
    missing = all_classes - classes

    print(f"{name:5s}: {len(classes)} classes")

    if missing:
        print(f"Missing classes: {sorted(list(missing))}")
    else:
        print("All classes present")

# 3. Distribution comparison
print("\nClass distribution (top differences):")

dist = pd.DataFrame({
    "train": train_df[target_col].value_counts(normalize=True),
    "val": val_df[target_col].value_counts(normalize=True),
    "test": test_df[target_col].value_counts(normalize=True),
}).fillna(0)

diff = (dist.max(axis=1) - dist.min(axis=1)).sort_values(ascending=False)

print("\nTop distribution shifts:")
print(diff.head(10))

# 4. sanity check
print("\nSanity checks:")
assert len(train_df) > 0, "Train set is empty!"
assert len(val_df) > 0, "Validation set is empty!"
assert len(test_df) > 0, "Test set is empty!"

assert len(all_classes) == df[target_col].nunique(), "Class mismatch detected!"

print("All sanity checks passed")

print("\n==========================================\n")

Dataset sizes:
train: 34,999
val  : 10,000
test : 5,001

Total: 50,000

Class coverage (number of classes per split):
train: 10 classes
   ✔ All classes present
val  : 10 classes
   ✔ All classes present
test : 10 classes
   ✔ All classes present

Class distribution (top differences):

Top distribution shifts:
group_id
1142    0.000206
232     0.000023
405     0.000023
489     0.000023
589     0.000023
769     0.000023
973     0.000023
1277    0.000023
2173    0.000023
2663    0.000023
dtype: float64

Sanity checks:
All sanity checks passed




In [ ]:
out_dir = "../data/processed/"

train_df.to_csv(out_dir + "train.csv", index=False)
val_df.to_csv(out_dir + "val.csv", index=False)
test_df.to_csv(out_dir + "test.csv", index=False)